# 05 – E′: tín hiệu hậu kỳ trong MER2024 (pilot cuối của nhánh E)
Notebook này đo xem nhạc nền, phụ đề in cứng (và cách tách chúng ra) ảnh hưởng thế nào tới nhận diện cảm xúc trên **MER2024 Train&Val (5.030 clip)**. Thiết kế và tiêu chí đã **đăng ký trước** trong docstring của `explore/editorial/mer_editorial.py`.

**Chỉ in và lưu số liệu tổng hợp** (`/kaggle/working/e_report.md`). Không in mẫu, transcript hay nhãn từng clip. Thư mục `/kaggle/working/e_cache/` chứa feature từng clip: **chỉ để trên Kaggle của bạn, không tải về, không chia sẻ** (điều khoản MER).

**Cài đặt notebook:** Accelerator = **GPU T4 x2** (code dùng 1 GPU); **Internet = On**.
1. *Add-ons → Secrets*: bật secret `HF_TOKEN` (giống notebook 04).
2. Nếu ô "Giải nén" báo `encrypted: True`, thêm secret `MER2024_ZIP_PW`. Lấy mật khẩu trong `README_AFTER_APPROVAL.md` của repo MER2024 trên Hugging Face (mở bằng tài khoản đã được duyệt). **Không dán mật khẩu vào chat hay vào code.**
3. *Save Version → Save & Run All*. Notebook tự dừng trước 10 giờ. Nếu chưa xong thì chạy lại, nó sẽ resume từ cache.

**Khi xong:** chỉ tải `e_report.md` rồi gửi cho Claude.

In [ ]:
HOURS = 9.0          # time budget for feature extraction (resumable)
KOUT, WORK = '/kaggle/working', '/tmp'
CACHE = f'{KOUT}/e_cache'

In [ ]:
import os, subprocess
sh = lambda cmd, check=True: subprocess.run(cmd, shell=True, check=check)
sh(f'rm -rf {WORK}/icme && git clone -q --depth 1 -b main https://github.com/ptrnghieu/icme.git {WORK}/icme')
print('code commit:', subprocess.run(f'git -C {WORK}/icme log -1 --format=%h_%cd', shell=True, capture_output=True, text=True).stdout)
sh('pip -q install -U huggingface_hub demucs easyocr', check=False)
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['HF_HOME'] = f'{WORK}/hf'
import torch; print('HF_TOKEN set:', bool(os.environ.get('HF_TOKEN')), '| cuda:', torch.cuda.is_available())

In [ ]:
# Download (only the files this pilot needs)
from huggingface_hub import hf_hub_download
D = f'{WORK}/mer'
for repo, fn in [('MERChallenge/MER2024', 'label-disdim.csv'), ('MERChallenge/MER2024', 'label-transcription.csv'),
                 ('MERChallenge/MER2024', 'video-labeled.zip'), ('MERChallenge/MER2025', 'track1_train_disdim.csv'),
                 ('MERChallenge/MER2025', 'track1_test_dis.csv')]:
    p = hf_hub_download(repo_id=repo, repo_type='dataset', filename=fn, local_dir=f'{D}/{repo.split("/")[1]}', token=os.environ['HF_TOKEN'])
    print(repo, fn, f'{os.path.getsize(p) / 1e9:.2f} GB')

In [ ]:
# Giải nén (mật khẩu, nếu có, chỉ đọc từ Kaggle secret, không in ra)
import zipfile, shutil
Z = f'{D}/MER2024/video-labeled.zip'; V = f'{WORK}/mer24_videos'
enc = any(zi.flag_bits & 0x1 for zi in zipfile.ZipFile(Z).infolist()[:50])
print('encrypted:', enc)
os.makedirs(V, exist_ok=True)
if enc:
    pw = UserSecretsClient().get_secret('MER2024_ZIP_PW')
    tool = '7z' if shutil.which('7z') else None
    if tool:
        r = subprocess.run(['7z', 'x', '-y', f'-p{pw}', f'-o{V}', Z], capture_output=True, text=True)
    else:
        r = subprocess.run(['unzip', '-q', '-o', '-P', pw, Z, '-d', V], capture_output=True, text=True)
    print('extract return code:', r.returncode)
    del pw
else:
    zipfile.ZipFile(Z).extractall(V)
import glob, pandas as pd
vids = {os.path.splitext(os.path.basename(p))[0] for p in glob.glob(f'{V}/**/*.*', recursive=True) if p.lower().endswith(('.mp4', '.avi', '.mkv'))}
names = set(pd.read_csv(f'{D}/MER2024/label-disdim.csv')['name'].astype(str))
print('video files:', len(vids), '| labelled names with a video:', len(names & vids), '/', len(names))

In [ ]:
# Feature extraction (GPU; resumable; budget HOURS)
args = ['python', f'{WORK}/icme/explore/editorial/mer_editorial.py', 'extract', '--videos', V, '--labels', f'{D}/MER2024/label-disdim.csv',
        '--cache', CACHE, '--hours', str(HOURS)]
subprocess.run(args, check=False)
print('parts:', len(glob.glob(f'{CACHE}/part_*.npz')))

In [ ]:
# Phân tích theo tiêu chí đã đăng ký trước -> e_report.md (chỉ số liệu tổng hợp)
args = ['python', f'{WORK}/icme/explore/editorial/mer_editorial.py', 'analyze', '--labels', f'{D}/MER2024/label-disdim.csv',
        '--trans', f'{D}/MER2024/label-transcription.csv', '--cache', CACHE, '--out', f'{KOUT}/e_report.md',
        '--mer25', f'{D}/MER2025/track1_train_disdim.csv', '--mer25test', f'{D}/MER2025/track1_test_dis.csv']
n_parts = len(glob.glob(f'{CACHE}/part_*.npz'))
if n_parts * 250 >= len(names):
    subprocess.run(args, check=False); print(open(f'{KOUT}/e_report.md').read())
else:
    print(f'extraction not finished ({n_parts} parts); re-run the notebook to resume')